# 🚀 Arquitetura 2 — CNN Moderna (BatchNorm + Dropout)

CNN profunda no estilo **VGG simplificado**: blocos de convoluções 3x3 empilhadas, **BatchNormalization** para estabilizar o treino e **Dropout** para regularização. Mais profunda e robusta que a LeNet.

## 1. Importações

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras import utils as utls
from keras.datasets import mnist
from keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

tf.random.set_seed(42)
np.random.seed(42)

## 2. Hiperparâmetros

In [ ]:
# Hiperparâmetros (idênticos ao exemplo LeNet original)
imageRows, imageCols = 28, 28
numClasses = 10
batchSize = 256
epochs = 10
cores = 1  # canais de cor (MNIST é escala de cinza)

## 3. Carga e pré-processamento do MNIST

In [ ]:
(XTrain, yTrain), (XTest, yTest) = mnist.load_data()
XTrain = XTrain.astype('float32') / 255.0
XTest = XTest.astype('float32') / 255.0
XTrain = XTrain.reshape(-1, imageRows, imageCols, cores)
XTest = XTest.reshape(-1, imageRows, imageCols, cores)

yTestLabels = yTest.copy()  # rótulos inteiros, usados na matriz de confusão
yTrain = utls.to_categorical(yTrain, numClasses)
yTest = utls.to_categorical(yTest, numClasses)
XTrain.shape, XTest.shape

## 4. Definição da arquitetura

In [ ]:
from keras.layers import (Conv2D, MaxPooling2D, BatchNormalization, Dropout,
                          Flatten, Dense)

inputShape = (imageRows, imageCols, cores)
model = Sequential(name='CNN_Moderna')

# Bloco 1
model.add(Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=inputShape))
model.add(BatchNormalization())
model.add(Conv2D(32, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Bloco 2
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Classificador
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(numClasses, activation='softmax'))
model.summary()

## 5. Compilação e treino (10 épocas)

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
historico = model.fit(XTrain, yTrain, batch_size=batchSize, epochs=epochs,
                      validation_data=(XTest, yTest))

## 6. Curva de acurácia

In [ ]:
f, ax = plt.subplots()
ax.plot(historico.history['accuracy'], 'o-')
ax.plot(historico.history['val_accuracy'], 'x-')
ax.legend(['Acurácia no Treinamento', 'Acurácia na Validação'], loc=0)
ax.set_title('Treinamento e Validação - Acurácia por Época')
ax.set_xlabel('Época')
ax.set_ylabel('Acurácia')

## 7. Matriz de confusão e relatório de classificação

In [ ]:
# Matriz de confusão no conjunto de teste
yPred = np.argmax(model.predict(XTest), axis=1)
cm = confusion_matrix(yTestLabels, yPred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True)
plt.title(f'Matriz de Confusão - {model.name}')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

print(classification_report(yTestLabels, yPred, digits=4))